# GRPO Training - Dialectical Materialism Alignment

Custom Group Relative Policy Optimization (no TRL/vLLM dependency).
Run inside Docker container `ml-training`.

**Expected duration**: 9-12 hours

## 1. Setup

In [ ]:
import json
import sys
from pathlib import Path

import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

sys.path.insert(0, "/app")

from src.student.grpo_config import GRPO_CONFIG
from src.student.rewards import (
    compute_directional_assertion,
    compute_format_reward,
    compute_length_reward,
    compute_dm_alignment_judge,
)

print("Imports OK")

In [ ]:
config = GRPO_CONFIG.copy()

# Override paths if needed
# config["base_model"] = "/mnt/c/Users/Guy/.unsloth/studio/exports/Qwen_Qwen3.5-9B_1779111714/checkpoint-330"
# config["output_dir"] = "checkpoints/lora_adapters/grpo_adapter"

print(f"Base model: {config['base_model']}")
print(f"Output dir: {config['output_dir']}")
print(f"Questions: {config['questions_path']}")
print(f"G: {config['grpo_g']}, Steps: {config['max_steps']}, LR: {config['learning_rate']}")
print(f"Beta (KL): {config['beta']}")
print(f"Reward weights: {config['reward_weights']}")

## 2. Load Model (NF4 + LoRA)

In [ ]:
from unsloth import FastLanguageModel

print(f"Loading model from {config['base_model']}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config["base_model"],
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=config["lora_rank"],
    lora_alpha=config["lora_alpha"],
    lora_dropout=config["lora_dropout"],
    target_modules=config["target_modules"],
)
model = FastLanguageModel.for_training(model)

print(f"LoRA applied: rank={config['lora_rank']}, alpha={config['lora_alpha']}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 3. Create Reference Model

In [ ]:
import copy
ref_model = copy.deepcopy(model)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False
print("Reference model created (frozen snapshot for KL penalty)")

## 4. Load Judge Model

In [ ]:
judge_model = None
judge_tokenizer = None

if config["reward_weights"].get("dm_alignment", 0) > 0:
    print(f"Loading judge model: {config['judge_model']}...")
    judge_model = AutoModelForCausalLM.from_pretrained(
        config["judge_model"],
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    judge_tokenizer = AutoTokenizer.from_pretrained(config["judge_model"])
    print(f"Judge model loaded on {judge_model.device}")
else:
    print("DM alignment reward disabled, skipping judge model")

## 5. Prepare Dataset

In [ ]:
with open(config["questions_path"], "r") as f:
    questions_data = json.load(f)

questions = [q["question"] for q in questions_data]
print(f"Loaded {len(questions)} questions")

prompts = []
for q in questions:
    chat = [{"role": "user", "content": q}]
    prompt_text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    prompts.append(prompt_text)

print(f"Dataset ready: {len(prompts)} samples")
print(f"\nSample prompt:\n{prompts[0][:300]}...")

## 6. Test Reward Functions

In [ ]:
from src.student.train_grpo import compute_rewards

test_completion = """<antThinking>Let me analyze this through material conditions...</antThinking>

The policy directly causes positive change for workers.

Material conditions drive outcomes in this context.

Power relationships are key to understanding this dynamic."""

scores = compute_rewards(
    [test_completion],
    config["reward_weights"],
    tokenizer,
    judge_model,
    judge_tokenizer,
)
print(f"Test reward score: {scores[0]:.4f}")

## 7. Train (Custom GRPO Loop)

In [ ]:
from src.student.train_grpo import train

train(config, config["base_model"], config["output_dir"])

## 8. Save Adapter

In [ ]:
print(f"Saving GRPO adapter to {config['output_dir']}...")
model.save_pretrained(config["output_dir"])
tokenizer.save_pretrained(config["output_dir"])
print("Done.")

## 9. Quick Smoke Test (Optional)

In [ ]:
test_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Why is income inequality increasing?"}],
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7)
generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(generated)